In [1]:
# Cell 1: Install dependencies

!pip install -q diffusers transformers accelerate safetensors

In [2]:
# Cell 2: Imports
import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path

In [ ]:
# Cell 3: Device setup

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Dtype:", dtype)

Device: cpu
Dtype: torch.float16


In [4]:
# Cell 4: Load model

MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)

pipe = pipe.to(device)

print("Model loaded successfully")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /home/tmedi/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /home/tmedi/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make

Model loaded successfully


In [7]:
# Cell 5: Generate function

def generate_images(category, num_images=4):
    
    prompt = f"a high quality photo of a {category}"
    
    images = pipe(
        [prompt] * num_images,
        num_inference_steps=25
    ).images
    
    return images

In [ ]:
# Cell 6: Try generation and show images (grid)

import matplotlib.pyplot as plt

category = "cow"

images = generate_images(category, num_images=4)

plt.figure(figsize=(10,10))

for i, img in enumerate(images):
    plt.subplot(2, 2, i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()

  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
# Cell 7: Save images

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

for i, img in enumerate(images):
    img.save(output_dir / f"{category}_{i}.png")

print("Saved images to outputs/")

In [ ]:
# Cell 8: Prompt-based generation

def generate_from_prompt(prompt, num_images=4):
    
    images = pipe(
        [prompt] * num_images,
        num_inference_steps=25
    ).images
    
    return images

In [ ]:
# Cell 9: Try prompt generation

prompt = "A futuristic city floating in the sky, ultra realistic, 4k"

images = generate_from_prompt(prompt, num_images=4)
# Cell 10: Show images

import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))

for i, img in enumerate(images):
    plt.subplot(2, 2, i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()

In [ ]:
# Cell 11: Style-based generation

def generate_with_style(prompt, style, num_images=4):
    
    styled_prompt = f"{prompt}, {style}"
    
    images = pipe(
        [styled_prompt] * num_images,
        num_inference_steps=25
    ).images
    
    return images

In [ ]:
# Cell 12: Try different styles

prompt = "a girl sitting on a chair"

styles = [
    "photorealistic",
    "cartoon style",
    "oil painting",
    "anime style"
]

images = []

for style in styles:
    img = generate_with_style(prompt, style, num_images=1)[0]
    images.append(img)

# Cell 13: Show style comparison

import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))

for i, img in enumerate(images):
    plt.subplot(2, 2, i+1)
    plt.imshow(img)
    plt.title(styles[i])
    plt.axis("off")

plt.show()